# Envio de Estoque WORDA Hub Sync

Documento escrito para o cliente **Mallon**.

Realiza o envio de todos os arquivos `csv` da pasta `.\data`, contendo estoque, para a API do Worda Hub.

Então, move os arquivos para a pasta `.\data\processed\`.

---

## Preparo do ambiente

Para esta versão, estaremos utilizando o Python 3.13.12. Os pacotes requeridos estão no arquivo requirements.txt.

In [1]:

! pip install --upgrade pip -q
! python --version


Python 3.13.12


In [2]:

# Para mover arquivos
import shutil

# Para realizar requisicoes HTTP
import requests

# Para manipular dados tabulares
import pandas as pd

# Para lidar arquivos e criar pastas
from pathlib import Path

# Para carregar .env no Windows, se necessario
import os
from dotenv import load_dotenv


In [3]:
# Variaveis de localizacao de diretorios
FOLDER_CURRENT = Path(".")
FOLDER_DATA = FOLDER_CURRENT / "data"
FOLDER_PROCESSED = FOLDER_DATA / "processed"

# Carrega variaveis de ambiente para a API
load_dotenv(FOLDER_CURRENT / ".env")
API_KEY = os.getenv("API_KEY", "exemplo_de_chave_secreta_aqui")
API_URL = os.getenv("API_URL", "http://localhost:8000/v1")
API_USERNAME = os.getenv("API_USERNAME", "string")
API_PASSWORD = os.getenv("API_PASSWORD", "string")


In [4]:
# Funcoes auxiliares

def list_csvs_in_folder(folder: Path) -> list[Path]:
    """Lista os arquivos .csv em uma pasta."""
    return list(folder.glob("*.csv"))

def grant_folders_are_created(folders: list[Path]) -> None:
    """Garante que as pastas existam, criando-as se necessario."""
    for folder in folders:
        folder.mkdir(exist_ok=True)

grant_folders_are_created([FOLDER_DATA, FOLDER_PROCESSED])

## Autenticação

Nosso sistema conta com autenticação bearer JWT.

Em função disso, o usuário deve realizar a autenticação e utilizar o token em todas as demais requisições.

In [5]:
# Coleta o token JWT

headers = {
    'accept': 'application/json',
    'Content-Type': 'application/json',
    'X-API-KEY': API_KEY,
}

payload = {
    "user_login": API_USERNAME,
    "user_pass": API_PASSWORD,
}

response = requests.post(
    url=f"{API_URL}/public/login",
    json=payload,
    headers=headers,
)

assert response.status_code == 200, f"Login failed with status code {response.status_code}"

auth_token = response.json().get("access_token")
auth_type = response.json().get("token_type")

assert auth_type == "bearer", "No access token found in response"


## Processa dados para o formato da API

Para o correto processamento dos dados por nossa API, os JSON enviados são validados e rejeitados caso não se adequem ao padrão.

As seguintes etapas transformam o `.csv` recebido no formato necessário, como descrito:

`

    {
    "emp_estoque": "string",
    "itens": [
        {
        "sku": "string",
        "title": "string",
        "quantidade": 0,
        "dias_estoque": 0,
        "preco_publico": 0
        }
    ]
    }


In [6]:
# Le todos os arquivos CSV da pasta de dados 
# e concatena em um único DataFrame

df = pd.DataFrame()
files_to_read = list_csvs_in_folder(FOLDER_DATA)

for csv in files_to_read:
    df_csv = pd.read_csv(
        csv,
        encoding="iso-8859-1",
    )
    df = pd.concat([df, df_csv], ignore_index=True)

df.head()


,EMPRESA,REVENDA,ITEM_ESTOQUE,ITEM_ESTOQUE_PUB,DES_ITEM_ESTOQUE,CLASS_ABC,CLASS_XYZ,QTD_CONTABIL,QTD_DISPONIVEL,CUSTO_MEDIO,PRECO_PUBLICO_ATUAL,DIAS_ESTOQUE
0,2,1,41194,1J0201048,CASQUILHO DISTANCIADOR,D,3,3.0,3.0,0.01,9.76,46083
1,2,1,65432,5U0035417B,"ALOJAMENTO TWEETER COLUNA ""A"" LE",D,3,1.0,1.0,0.01,22.39,46083
2,2,1,48880,211415753,ANEL DE VEDACAO,D,3,1.0,1.0,0.01,6.07,46083
3,2,1,48879,2114157491,VEDACAO,D,3,1.0,1.0,0.01,21.02,46083
4,2,1,12295,036109309F,TUCHO,D,3,16.0,16.0,31.51,99.00,5861


In [7]:
# Aplica as transformações necessárias para 
# adequar o DataFrame ao formato esperado pela API

empresa_ml_map = {
    2: 226055463,
}

rename_columns = {
    "ITEM_ESTOQUE_PUB": "sku",
    "DES_ITEM_ESTOQUE": "title",
    "QTD_DISPONIVEL": "quantidade",
    "DIAS_ESTOQUE": "dias_estoque",
    "PRECO_PUBLICO_ATUAL": "preco_publico"
}

dtypes = {
    "sku": str,
    "title": str,
    "quantidade": int,
    "dias_estoque": int,
    "preco_publico": float,
    "emp_estoque": str
}

df.rename(columns=rename_columns, inplace=True)
df["emp_estoque"] = df["EMPRESA"].map(empresa_ml_map)

final_columns = ["emp_estoque"] + list(rename_columns.values())
df.drop(
    columns=[col for col in df.columns if col not in final_columns]
    , inplace=True
)

df = df.astype(dtypes, errors="ignore")
df.head()


,sku,title,quantidade,preco_publico,dias_estoque,emp_estoque
0,1J0201048,CASQUILHO DISTANCIADOR,3,9.76,46083,226055463
1,5U0035417B,"ALOJAMENTO TWEETER COLUNA ""A"" LE",1,22.39,46083,226055463
2,211415753,ANEL DE VEDACAO,1,6.07,46083,226055463
3,2114157491,VEDACAO,1,21.02,46083,226055463
4,036109309F,TUCHO,16,99.00,5861,226055463


In [8]:
# Transforma o DataFrame para o formato esperado pela API,
# onde cada linha representa um estoque de uma empresa, e a coluna "itens" é
# uma lista de dicionários, cada um representando um item em estoque

df["itens"] = df.apply(
    lambda row: dict(row),
    axis=1
)

cols_to_drop = [col for col in df.columns if col not in ["emp_estoque", "itens"]]
df.drop(
    columns=cols_to_drop,
    inplace=True
)

df.head()


,emp_estoque,itens
0,226055463,"{'sku': '1J0201048', 'title': 'CASQUILHO DISTA..."
1,226055463,"{'sku': '5U0035417B', 'title': 'ALOJAMENTO TWE..."
2,226055463,"{'sku': '211415753', 'title': 'ANEL DE VEDACAO..."
3,226055463,"{'sku': '2114157491', 'title': 'VEDACAO', 'qua..."
4,226055463,"{'sku': '036109309F', 'title': 'TUCHO', 'quant..."


In [9]:
# Agrupa por empresa e transforma a 
# coluna "itens" em uma lista de dicionários

df = df.groupby("emp_estoque").agg(
    lambda x: list(x)
).reset_index(drop=False)
df.head()



,emp_estoque,itens
0,226055463,"[{'sku': '1J0201048', 'title': 'CASQUILHO DIST..."


In [10]:
# Transforma o DataFrame em uma lista de dicionários, 
# onde cada dicionário representa um estoque de uma empresa

payloads = df.to_dict(orient="records")
print(str(payloads)[:100])


[{'emp_estoque': '226055463', 'itens': [{'sku': '1J0201048', 'title': 'CASQUILHO DISTANCIADOR', 'qua


## Envia para API

Realiza o envio dos dados para a API.

Apenas se deve considerar como sucesso se o retorno possuir código HTTP 200.

O retorno contém a contagem de itens processados corretamente.

Exemplo de retorno:

    {
        'status': 'Processamento concluído', 
        'inseridos': 2281, 
        'atualizados_qtd': 0, 
        'atualizados_info': 0, 
        'ignorados': 0, 
        'skus_inseridos': ['1J0201048', ...], 
        'skus_atualizados': [...], 
        'skus_atualizados_info': [...], 
        'skus_ignorados': [...], 
        'erros': [...]
    }


In [11]:
# Envia o payload para a API

endpoint = f"{API_URL}/private/shopee/estoque/atualizar"

# Apenas uma empresa, para demonstracao
payload = payloads[0]

response = requests.post(
    url=endpoint,
    json=payload,
    headers={
        **headers,
        "Authorization": f"{auth_type.capitalize()} {auth_token}"
    },
)

print(response.status_code)
print(str(response.json())[:1000])

assert response.status_code == 200, (
    f"Failed to send payload for emp_estoque {payload['emp_estoque']}. "
    f"Status code: {response.status_code}"
) 

print(
    f"Payload for emp_estoque {payload['emp_estoque']} "
    "sent successfully."
)


200
{'status': 'Processamento concluído', 'inseridos': 0, 'atualizados_qtd': 0, 'atualizados_info': 0, 'ignorados': 2281, 'skus_inseridos': [], 'skus_atualizados': [], 'skus_atualizados_info': [], 'skus_ignorados': ['1J0201048', '5U0035417B', '211415753', '2114157491', '036109309F', '5Z6827517EE', '5K0698231', '1Y0837111', '1J0609707A', '7L0498103AX', '6R0803105', '6KE010483B', '2H7885500C', '2H0598160', 'N90707604', '5K0837350B', '2H0609721D', '5Q0412047A', '155253115', 'N10155906', '2H0906376C', '377201193B', 'N10770701', '02T311285Q', 'N91039802', '036907601E', 'N91130801', '06G129748', 'N10328002', '5Z0959565C9B9', '06A121119', 'N91073401', 'N91168901', 'N90888502', '5N0807651B', '5N0807109F', 'N10347003', '1K0823480', '5K0959591', '2HH8074909B9', '036145276', 'N91143301', '5C5012003EP', 'WHT002109', '089409069', 'N10272302', 'WHT005430', 'N0444076', 'WHT000036', 'N90683306', '030121043C', 'N91201001', '7L0407357B', 'N10209009', '131821143Z', '6Q0711067A', 'N91226101', 'N10435508',

## Marca arquivos como processados

Por fim, marca os arquivos como processados.

Deve-se atentar que aqui são marcados todos os arquivos, mas, para esta PoC, apenas uma das empresas se está enviando. Caso haja mais de uma, é importante revisar este processo.



In [12]:

for file in files_to_read:
    
    try:
        destination = FOLDER_PROCESSED / file.name
        shutil.move(src=file, dst=destination)
        print(f"Moved file {file.name} to {destination}")
    
    except Exception as e:
        print(f"Failed to move file {file.name}: {e}")
    

Moved file estoque_diario.csv to data\processed\estoque_diario.csv


---

Miguel Zanchettin, Worda.
Março de 2026.